## Langchain

In [ ]:
!pip install langchain langchain-groq langchain-core -q

In [ ]:
from google.colab import userdata
from langchain_groq import ChatGroq

In [ ]:
my_api_key = userdata.get('groq_api')

In [ ]:
llm = ChatGroq(
    model = 'llama-3.3-70b-versatile',
    temperature = 0.7,
    api_key = my_api_key
)

In [ ]:
llm.invoke('What is the capital of India?')

AIMessage(content='The capital of India is New Delhi.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 9, 'prompt_tokens': 42, 'total_tokens': 51, 'completion_time': 0.011843543, 'completion_tokens_details': None, 'prompt_time': 0.001181426, 'prompt_tokens_details': None, 'queue_time': 0.04792785, 'total_time': 0.013024969}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_f8b414701e', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019eb6fb-52ba-7831-8636-4aa5cd7e3d0d-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 42, 'output_tokens': 9, 'total_tokens': 51})

In [ ]:
# We cannot trust our users to write good prompts
# We should give them prompt templates with good prompts and fill in the blank

from langchain_core.prompts import PromptTemplate

# 1. Create the Template
# We use {brackets} to define the information we expect from the user
template = """
You are an expert copywriter.
Write a catchy 5-word slogan for a company that makes {product}
"""

In [ ]:
prompt = PromptTemplate.from_template(template)

In [ ]:
text = prompt.format(product='Flying Car')
print(text)


You are an expert copywriter.
Write a catchy 5-word slogan for a company that makes Flying Car



In [ ]:
# The Output Parser (Parser means to interpret and extract useful info)
# Raw LLM output contains lots of extra stuff
# StrOutputParser (String Output Parser)

# We are using a 'String' output parser, because output of LLM
# is always String (sentences/words)

from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()

In [ ]:
# prompt -> llm -> parser
# Langchain is called Lang'Chain' because it can 'chain' all 3 into one 'chain'

chain = prompt | llm | parser # LCEL Langchain Expression Language

# Running the chain
result = chain.invoke({"product": "spicy ice cream"})
print(result)

#

"Savoring Heat in Every Bite"


In [ ]:
# Mini-Project 9: "The Title Generator"

# Goal: Create a sequential Chain.
# Step 1 generates a YouTube title from a topic;
# Step 2 writes a video description for it.

# Step 1: The Title Generator Chain
template = """
You are a viral Youtube Creator. Give me exactly ONE highly clickable title for
a video about {topic}. Do not use quotes
"""

title_prompt = PromptTemplate.from_template(template)

title_chain = title_prompt | llm | StrOutputParser()


# Step 2: The Description Chain
template = """
Write a 2 sentence YouTube description for a video titled: {title}
"""

desc_prompt = PromptTemplate.from_template(template)
desc_chain = desc_prompt | llm | StrOutputParser()

In [ ]:
def youtube_workflow(video_topic):
    print(f"🎬 Starting workflow for topic: '{video_topic}'...")

    # Run the first chain to get the title
    generated_title = title_chain.invoke({"topic": video_topic})

    print(f"\n✨ Step 1 (Title): {generated_title}")

    # Pass the title to the second chain
    generated_desc = desc_chain.invoke({"title": generated_title})

    print(f"\n📝 Step 2 (Description): {generated_desc}")

In [ ]:
# Practice Challenge: "The Translator Pipeline"
# Scenario: You work for an international news agency.
# You need a pipeline that takes a raw English headline, translates it to French,
# and then translates that French version into Spanish to check for consistency.

# Your Task:

# Create a french_chain that takes an {english_text} and outputs French.

# Create a spanish_chain that takes a {french_text} and outputs Spanish.

# Write a function translation_workflow(text) that chains them together and prints both steps.

In [ ]:
french_template = """
Translate the following text from English to French.
Only return the translated text, nothing else. Don't give me the quotes in the output.
English Text: {english_text}
"""
french_prompt = PromptTemplate.from_template(french_template)
french_chain = french_prompt | llm | StrOutputParser()


spanish_template = """
Translate the following text from French to Spanish.
Only return the translated text, nothing else. Don't give me the quotes in the ouput.
French Text: {french_text}
"""


spanish_prompt = PromptTemplate.from_template(spanish_template)
spanish_chain = spanish_prompt | llm | StrOutputParser()
def translation_workflow(text):
    french_result = french_chain.invoke({"english_text": text})
    print(f"French: {french_result}")


    spanish_result = spanish_chain.invoke({"french_text": text})
    print(f"Spanish: {spanish_result}")


translation_workflow("Scientists discover water on Mars")

French: Les scientifiques découvrent de l'eau sur Mars
Spanish: Científicos descubren agua en Marte


# Let's start creating AI Agents!

In [ ]:
pip install langchain-groq -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 7.5 MB/s eta 0:00:00


In [ ]:
from google.colab import userdata

In [ ]:
my_api_key = userdata.get('groq_api')

In [ ]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model = 'llama-3.3-70b-versatile',
    api_key = my_api_key,
    temperature = 0.5 # 0 for factual answer
)

In [ ]:
# 70 billion parameters?
# It means to predict 'one' next word the LLM is doing 70,000,000,000 calculations!!

In [ ]:
llm.invoke('Write a story on black holes')

AIMessage(content='In the depths of space, there existed a mysterious and elusive phenomenon known as a black hole. It was a region in space where the gravitational pull was so strong that nothing, not even light, could escape once it got too close. The black hole was like a cosmic vacuum cleaner, sucking in everything that dared to venture near.\n\nA team of scientists, led by the brilliant and fearless Dr. Maria Rodriguez, had been studying the black hole for years. They had been tracking its movements, monitoring its activity, and trying to learn more about its secrets. The team had been on a mission to explore the black hole up close, to gather data and conduct experiments that would help them better understand this enigmatic phenomenon.\n\nAs they approached the black hole, the team could feel the gravity pulling them in. The ship was equipped with state-of-the-art technology, but even that was no match for the black hole\'s incredible power. The ship was being pulled closer and c

In [ ]:
result = llm.invoke('What is 5 * 4. Give me the numerical answer only')
result.content

'20'

In [ ]:
result = llm.invoke('What is 10 * 20. Give me the numerical answer only')
result.content

'200'

In [ ]:
result = llm.invoke('Multiply 952124 and 123991. Give me the numerical answer only')
result.content

# 118054806884

'118253879084'

In [ ]:
# Train a new LLM which can do math perfectly
# Calculator can do calculations, but they don't understand your query, they cannot generate text
# LLMs can understand query and generate text

In [ ]:
# Docstring. Every tool to be given to LLM should contain a Docstring (Description of the tool)
@tool
def multiply(a: int, b: int) -> int:
    """Multiplies two integers together. Use this tool whenever you need to perform calculation"""

    print("⭐Multiply function called")
    return a*b

In [ ]:
@tool
def divide(a: int, b: int) -> float:
    """Divides two integers together. Use this tool whenever you need to perform division"""

    print("⭐Divide function called")
    return a/b

In [ ]:
llm = llm.bind_tools([multiply, divide])

In [ ]:
result = llm.invoke('Multiply 952124 and 123991. Give me the numerical answer only')
result.content

''

In [ ]:
result.tool_calls

[{'name': 'multiply',
  'args': {'a': 952124, 'b': 123991},
  'id': 'nc25f2re9',
  'type': 'tool_call'}]

In [ ]:
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph.message import add_messages

In [ ]:
class State(TypedDict):
    messages: Annotated[list, add_messages] # Common list of messages

In [ ]:
def assistant_node(state: State):
    # Read the old conversation history
    current_messages = state['messages']

    # Ask the LLM what to do
    response = llm.invoke(current_messages)

    # Store this new response in our message list
    return {'messages': [response]}

In [ ]:
tools = [multiply, divide]

In [ ]:
from langgraph.prebuilt import ToolNode
tool_node = ToolNode(tools)

In [ ]:
from typing import Literal
from langgraph.graph import END

def should_continue(state: State) -> Literal['tools', END]:
    messages = state['messages']

    last_message = messages[-1] # -1 for last message

    if last_message.tool_calls:
        return 'tools'

    else:
        return END

In [ ]:
from langgraph.graph import StateGraph, START

# Create a blank graph
builder = StateGraph(State)

In [ ]:
builder.add_node("assistant", assistant_node)
builder.add_node("tools", tool_node)

In [ ]:
builder.add_edge(START, "assistant")

In [ ]:
builder.add_conditional_edges(
    "assistant",
    should_continue
)

In [ ]:
builder.add_edge("tools", "assistant")

In [ ]:
graph = builder.compile()

In [ ]:
# 1. Define the user's complex request
user_query = "What is 952124 * 123991? Finally divide the answer by 10."
# 2. Run the Graph
# We stream the steps so we can see the "thinking" process live
initial_state = {"messages": [("user", user_query)]}

print(f"User: {user_query}\n")
print("--- STARTING AGENT LOOP ---")

for step in graph.stream(initial_state):
    # Print the name of the node that just finished running
    node_name = list(step.keys())[0]
    print(f"📍 Node '{node_name}' finished.")

    # Optional: Print the actual output to see what happened inside
    # print(step)

print("--- END ---")

# 3. Print the final result
final_response = graph.invoke(initial_state)
print("\n🤖 Final Answer:")
print(final_response["messages"][-1].content)

User: What is 952124 * 123991? Finally divide the answer by 10.

--- STARTING AGENT LOOP ---
📍 Node 'assistant' finished.
⭐Multiply function called
⭐Divide function called
📍 Node 'tools' finished.
📍 Node 'assistant' finished.
--- END ---
⭐Multiply function called
⭐Divide function called

🤖 Final Answer:
The result of 952124 * 123991 is 118054806884, and then dividing that by 10 gives 11805480688.4.


In [ ]:
# Add a divide tool
# And ask What is 200 * 40. Then divide the result by 10.

In [ ]:
!pip install duckduckgo-search langchain-community -q

In [ ]:
pip install -U ddgs

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.6/70.6 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.7/161.7 kB 11.3 MB/s eta 0:00:00


In [ ]:
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_core.tools import tool
# Initialize the underlying search engine
ddg_search = DuckDuckGoSearchRun()

@tool
def web_search(query: str) -> str:
    """Searches the web for current events, facts, or up-to-date information. Use this tool to answer general knowledge questions."""
    print(f"⭐Web Search function called for: '{query}'")
    return ddg_search.invoke(query)

# Now add your custom function to the tools list
tools = [multiply, divide, web_search]

In [ ]:
from langgraph.prebuilt import ToolNode

# 1. Bind the updated tools list to the LLM
llm_with_tools = llm.bind_tools(tools)

# 2. Recreate the tool node with the new tools list
tool_node = ToolNode(tools)

# 3. Update the assistant node to use the newly bound LLM
def assistant_node(state: State):
    current_messages = state['messages']
    # Use llm_with_tools here instead of the raw llm
    response = llm_with_tools.invoke(current_messages)
    return {'messages': [response]}

In [ ]:
from langgraph.graph import StateGraph, START

# Rebuild the graph
builder = StateGraph(State)
builder.add_node("assistant", assistant_node)
builder.add_node("tools", tool_node)

builder.add_edge(START, "assistant")
builder.add_conditional_edges("assistant", should_continue)
builder.add_edge("tools", "assistant")

graph = builder.compile()

# Test the Web Search Agent
user_query = "Who is the current CEO of OpenAI and what is 500 divided by 20? Answer both question"
initial_state = {"messages": [("user", user_query)]}

print("--- STARTING AGENT LOOP ---")
for step in graph.stream(initial_state):
    node_name = list(step.keys())[0]
    print(f"📍 Node '{node_name}' finished.")
print("--- END ---\n")

print("🤖 Final Answer:")
print(graph.invoke(initial_state)["messages"][-1].content)

--- STARTING AGENT LOOP ---


BadRequestError: Error code: 400 - {'error': {'message': "Failed to call a function. Please adjust your prompt. See 'failed_generation' for more details.", 'type': 'invalid_request_error', 'code': 'tool_use_failed', 'failed_generation': '<function=web_search{"query": "current CEO of OpenAI"}</function>\n'}}

In [ ]:
# Practice Challenge: "The Word Counter Tool"
# Scenario: LLMs are famously terrible at counting letters and words because they read "Tokens," not letters.
# Let's fix that.

# Create a new tool called word_counter

In [ ]:
llm = ChatGroq(
    model = 'llama-3.3-70b-versatile',
    api_key = my_api_key,
    temperature = 2 # 0 for factual answer
)

llm.invoke('Choose a random number between 1 to 10').content

'The random number is: 7'

In [ ]:
# Give a tool to the LLM to choose a truly random number

In [ ]:
llm = ChatGroq(
    model = 'llama-3.3-70b-versatile',
    api_key = my_api_key,
    temperature = 0 # 0 for factual answer
)

In [ ]:
def multiply(a: int, b: int) -> int:
    """Multiplies two integers together. Use this tool whenever you need to perform calculation"""

    print("⭐Multiply function called")
    return a*b

def divide(a: int, b: int) -> float:
    """Divides two integers together. Use this tool whenever you need to perform division"""

    print("⭐Divide function called")
    return a/b

ddg_search = DuckDuckGoSearchRun()

def web_search(query: str) -> str:
    """Searches the web for current events, facts, or up-to-date information. Use this tool to answer general knowledge questions."""
    print(f"⭐Web Search function called for: '{query}'")
    return ddg_search.invoke(query)


tools = [multiply, divide, web_search]

In [ ]:
system_prompt_text = """
You are a helpful assistant. Use the correct tools in the correct order to solve the user's query
"""

In [ ]:
from langchain_core.messages import SystemMessage
from langgraph.prebuilt import create_react_agent

system_message = SystemMessage(content=system_prompt_text)

agent = create_react_agent(llm, tools, prompt=system_message)

/tmp/ipykernel_3406/3789239012.py:6: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(llm, tools, prompt=system_message)


In [ ]:
result = agent.invoke({"messages": [("user", "What is 952124 * 123991?")]})
print(result['messages'][-1].content)

⭐Multiply function called
The result of 952124 * 123991 is 118054806884.


In [ ]:
# Challenge 1: The "News Anchor"
# Scenario: You want a quick summary of current events.
# Task: Ask the agent to:
# "Search for the latest news about Artificial Intelligence today and summarize the top 3 points in a bulleted list."

In [ ]:
# Challenge 2: The "Data Analyst"
# Scenario: You need to do math based on live, real-world data.
# Task: Ask the super_agent to:
# "Search for the current population of the United States. Then, use the calculator to find out what 10% of that population is."

In [ ]:
# Practice Challenge 3: The "Game Master" (Fixing Randomness)
# The Flaw: If you ask an LLM to "Pick a random number between 1 and 10," it does not actually generate a random number. It just predicts the most likely number based on human text data (which is usually 7).

# The Solution: We need to build a Python tool that uses the random library to force true, mathematical randomness, and give it to our LangGraph Agent.

# Your Task:

# Import the random library (import random).

# Create a new tool called roll_dice using the @tool decorator.

# The tool should take sides: int as an input (e.g., 6 for a standard die, 20 for a D&D die).

# Inside the tool, use random.randint(1, sides) to get the result. Return it as a string.

# Combine this new tool with the calculator tool we built earlier into a new game_agent using create_react_agent.

# Ask the agent: "Roll a 20-sided die. Then, multiply the result by 5."

In [ ]:
pip install langchain-groq -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 3.0 MB/s eta 0:00:00


In [ ]:
from google.colab import userdata

my_api_key = userdata.get('groq_api')

from langchain_groq import ChatGroq

llm = ChatGroq(
    model = 'llama-3.3-70b-versatile',
    api_key = my_api_key,
    temperature = 0.0 # 0 for factual answer
)

In [ ]:
from langchain_core.tools import tool
import random

@tool
def multiply(a: int, b: int) -> int:
    """Multiplies two integers together. Use this tool whenever you need to perform calculation"""

    print("⭐Multiply function called")
    return a*b

@tool
def roll_dice(sides: int) -> str:
    """Use this tool to roll a die. You can decide the number of sidees the die has"""
    print("⭐Roll Die function called")
    num = random.randint(1, sides)
    return str(num)

tools = [multiply, roll_dice]

In [ ]:
system_prompt_text = """
You are a helpful assistant. Use the correct tools in the correct order to solve the user's query
"""

from langchain_core.messages import SystemMessage
from langgraph.prebuilt import create_react_agent

system_message = SystemMessage(content=system_prompt_text)

game_agent = create_react_agent(llm, tools, prompt=system_message)

/tmp/ipykernel_577/3711586590.py:10: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  game_agent = create_react_agent(llm, tools, prompt=system_message)


In [ ]:
result = game_agent.invoke({"messages": [("user", "Roll a 20-sided die. Then, multiply the result by 5.")]})
print(result['messages'][-1].content)

⭐Multiply function called
⭐Roll Die function called
The result of rolling a 20-sided die is 7. Then, multiplying 7 by 5 gives 35.


In [ ]:
# 1. Define the Anomaly Detection Tool
@tool
def detect_anomaly(temperature: float) -> str:
    """
    Use this tool to check if a server temperature reading is an anomaly.
    Input should be the temperature as a number (float).
    """
    if temperature > 90.0:
        return "🚨 ANOMALY DETECTED: Critical Failure Risk!"
    else:
        return "✅ Status Normal."





In [ ]:
system_prompt_text = """
You are a helpful assistant. Use the correct tools in the correct order to solve the user's query. Only return directly whatever the tools give.
"""

from langchain_core.messages import SystemMessage
from langgraph.prebuilt import create_react_agent

system_message = SystemMessage(content=system_prompt_text)

monitor_agent = create_react_agent(llm, tools=[detect_anomaly], prompt=system_message)

/tmp/ipykernel_577/3098052166.py:10: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  monitor_agent = create_react_agent(llm, tools=[detect_anomaly], prompt=system_message)


In [ ]:
# 3. Ask the Question
question = "Check the server. The current temperature is 94.5 degrees"
print(f"👤 Question: {question}")

# 4. Run the Graph
result_state = monitor_agent.invoke({
    "messages": [("user", question)]
})

final_answer = result_state["messages"][-1].content
print(f"🤖 Answer: {final_answer}")

👤 Question: Check the server. The current temperature is 94.5 degrees


KeyboardInterrupt: 

In [ ]:
# The Autonomous Market Researcher Agent
# Scrape the website

In [ ]:
!pip install duckduckgo-search langchain_community langgraph beautifulsoup4 requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 39.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 44.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 554.9/554.9 kB 24.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 75.9 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.4.3
    Uninstalling langchain-core-1.4.3:
      Successfully uninstalled langchain-core-1.4.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is inco

In [ ]:
pip install -U ddgs

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.6/70.6 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.7/161.7 kB 9.8 MB/s eta 0:00:00


In [ ]:
import requests
from bs4 import BeautifulSoup
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_core.tools import tool

# Web Scraper
@tool
def scrape_websites(url: str) -> str:
    """
    Use this tool to read the full text of a specific webpage.
    Input MUST be a valid URL starting with http or https.
    """
    try:
        # 1. Fetch the webpage
        response = requests.get(url, timeout=10)

        # 2. Scrape the HTML
        soup = BeautifulSoup(response.text, 'html.parser')

        # 3.Extract only the paragraph text (to avoid code and menu random text)
        paragraphs = soup.find_all('p')
        text_content = " ".join([p.text for p in paragraphs])

        # 4 Limit the size of the paragraph
        return text_content[:3000]

    except Exception as e:
        return f"Error in scraping website: {e}"

ddg_search = DuckDuckGoSearchRun()

@tool
def web_search(query: str) -> str:
    """Searches the web for current events, facts, or up-to-date information. Use this tool to answer general knowledge questions."""
    print(f"⭐Web Search function called for: '{query}'")
    return ddg_search.invoke(query)


tools = [scrape_websites, web_search]


In [ ]:
from google.colab import userdata

my_api_key = userdata.get('groq_api')

from langchain_groq import ChatGroq

llm = ChatGroq(
    model = 'llama-3.3-70b-versatile',
    api_key = my_api_key,
    temperature = 0.0 # 0 for factual answer
)

In [ ]:
# 2. The Master Instruction (Fulfilling the Capstone Rubric)
system_instruction = """
You are a Senior Investment Analyst.
When asked to research a company, you must follow this exact process:

1. PLAN & RESEARCH: Use the `web_search` tool to find the most recent news about the company.
2. SCRAPE: Find at least ONE interesting URL from the search results and use the `scrape_website` tool to read the article.
3. ANALYZE: Review the data to identify "Bullish" (positive growth) and "Bearish" (negative risk) sentiments.
4. REPORT: Generate a final Investment Memo formatted strictly in Markdown.

Your final output MUST look exactly like this structure:

# Investment Memo: [Company Name]
**Date of Research:** [Current Date]

## 1. Company Overview & Recent News
[Summary of what the company is doing right now]

## 2. Bullish Sentiment 📈
* [Positive point 1]
* [Positive point 2]

## 3. Bearish Sentiment 📉
* [Negative point 1]
* [Negative point 2]

## 4. Final Analyst Conclusion
[Your final verdict]
"""

In [ ]:

from langchain_core.messages import SystemMessage
from langgraph.prebuilt import create_react_agent

system_message = SystemMessage(content=system_instruction)

market_researcher = create_react_agent(llm, tools, prompt=system_message)

/tmp/ipykernel_7259/2765798617.py:6: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  market_researcher = create_react_agent(llm, tools, prompt=system_message)


In [ ]:
# The Input (Step 1 of the Curriculum)
company_to_research = "RajeshExports"

print(f"🚀 Launching Research Agent for: {company_to_research}\n")
print("This may take 30-60 seconds as the AI surfs the web...\n")

# Execute the Graph
result_state = market_researcher.invoke({
    "messages": [("user", f"Write an investment memo for {company_to_research}.")]
})

# Extract and print the final Markdown report
final_report = result_state["messages"][-1].content

print("==========================================")
print(final_report)
print("==========================================")

🚀 Launching Research Agent for: RajeshExports

This may take 30-60 seconds as the AI surfs the web...

⭐Web Search function called for: 'Rajesh Exports latest news'
# Investment Memo: Rajesh Exports
**Date of Research:** June 16, 2026

## 1. Company Overview & Recent News
Rajesh Exports is a prominent player in the gold jewelry export space. The company has faced headwinds from fluctuating gold prices and muted global demand for jewelry. Recently, the company's shares rebounded 5% after seven days of losses. However, the company has been barred by SEBI over revenue concerns, and it has denied any wrongdoing.

## 2. Bullish Sentiment 
* The company's shares have rebounded 5% after seven days of losses, indicating a potential recovery in the stock price.
* Rajesh Exports is a leading player in the gold jewelry export space, which could provide a competitive advantage.

## 3. Bearish Sentiment 
* The company has been barred by SEBI over revenue concerns, which could impact its financial p

In [ ]:
# "The Competitor Analysis"
# To prove your system is robust, let's change the input to require a slightly more complex comparative search.

# Your Task:
# Modify the company_to_research variable in Step 3 to:
# "NVIDIA vs. AMD"